# DistilBERT Fine-Tuning

Fine-tune a pre-trained **DistilBERT** transformer on the fake-news dataset
(transfer learning). The fine-tuned model is saved so it can be compared
against the baseline and used in the demo app.

**Dataset:** `GonzaloA/fake_news`. Labels: `0 = FAKE`, `1 = REAL`.

> Training runs on GPU automatically if available, otherwise CPU. The sample
> sizes below keep CPU training to a few minutes; increase them for higher
> accuracy if you have a GPU.

In [ ]:
import json, time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'distilbert-base-uncased'
TRAIN_SIZE = 4000     # increase on GPU for better results
TEST_SIZE = 2000
MAX_LEN = 192
EPOCHS = 2
BATCH = 16
LR = 5e-5
RANDOM_STATE = 42

RESULTS_DIR = Path('..') / 'results'
(RESULTS_DIR / 'models').mkdir(parents=True, exist_ok=True)
LABEL_NAMES = {0: 'FAKE', 1: 'REAL'}
print('Device:', DEVICE)

## 1. Load and sample the data

A random sample keeps training fast. The same `title + text` content field is
used as in the baseline notebook.

In [ ]:
ds = load_dataset('GonzaloA/fake_news')
train = ds['train'].shuffle(seed=RANDOM_STATE).select(range(TRAIN_SIZE))
test = ds['test'].shuffle(seed=RANDOM_STATE).select(range(TEST_SIZE))

def to_content(batch):
    title = batch.get('title') or ''
    text = batch.get('text') or ''
    return {'content': (str(title) + '. ' + str(text)).strip()}

train = train.map(to_content)
test = test.map(to_content)
print('Train sample:', len(train), '| Test sample:', len(test))

## 2. Tokenise

DistilBERT uses WordPiece tokenisation. Each article is truncated/padded to a
fixed length.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(dataset):
    enc = tokenizer(list(dataset['content']), truncation=True, padding='max_length',
                    max_length=MAX_LEN, return_tensors='pt')
    labels = torch.tensor(list(dataset['label']))
    return enc['input_ids'], enc['attention_mask'], labels

train_ids, train_mask, train_y = encode(train)
test_ids, test_mask, test_y = encode(test)
print('Tokenised train:', train_ids.shape)

## 3. Data loaders

In [ ]:
train_loader = DataLoader(TensorDataset(train_ids, train_mask, train_y),
                          batch_size=BATCH, shuffle=True)
test_loader = DataLoader(TensorDataset(test_ids, test_mask, test_y),
                         batch_size=BATCH)
print('Batches per epoch:', len(train_loader))

## 4. Model and training loop

Load the pre-trained DistilBERT with a classification head and fine-tune it with
the AdamW optimiser.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

model.train()
for epoch in range(EPOCHS):
    t0 = time.time()
    total = 0.0
    for input_ids, attention_mask, labels in train_loader:
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        out.loss.backward()
        optimizer.step()
        total += out.loss.item()
    print(f'Epoch {epoch + 1}/{EPOCHS} | avg loss {total / len(train_loader):.4f} | {time.time() - t0:.0f}s')

## 5. Evaluate

In [ ]:
model.eval()
preds = []
with torch.no_grad():
    for input_ids, attention_mask, labels in test_loader:
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        preds.extend(logits.argmax(dim=1).cpu().numpy())

preds = np.array(preds)
y_true = test_y.numpy()

distilbert_metrics = {
    'accuracy': accuracy_score(y_true, preds),
    'precision': precision_score(y_true, preds),
    'recall': recall_score(y_true, preds),
    'f1': f1_score(y_true, preds),
}
print({k: round(v, 4) for k, v in distilbert_metrics.items()})

cm = confusion_matrix(y_true, preds)
plt.figure(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=[LABEL_NAMES[0], LABEL_NAMES[1]],
            yticklabels=[LABEL_NAMES[0], LABEL_NAMES[1]])
plt.title('Confusion matrix - DistilBERT')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout(); plt.show()

print(classification_report(y_true, preds, target_names=[LABEL_NAMES[0], LABEL_NAMES[1]]))

## 6. Save model and metrics

In [ ]:
with open(RESULTS_DIR / 'distilbert_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(distilbert_metrics, f, indent=2)

save_dir = RESULTS_DIR / 'models' / 'distilbert'
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print('Saved DistilBERT model to', save_dir.resolve())
print('F1 =', round(distilbert_metrics['f1'], 4))